# Experiment 1 V2 — Together AI
End-to-end 50-intent experiment.

In [ ]:
!pip -q install -r /content/experiment-1-v2-together/requirements.txt

In [ ]:
from pathlib import Path
import sys,os
ROOT=Path('/content/experiment-1-v2-together'); sys.path.insert(0,str(ROOT))

## Together API key from Colab Secrets

In [ ]:
from google.colab import userdata
os.environ['TOGETHER_API_KEY']=userdata.get('TOGETHER_API_KEY')
assert os.environ.get('TOGETHER_API_KEY')

## Dataset

In [ ]:
import pandas as pd
dataset=ROOT/'data/raw/intents_50.csv'; df=pd.read_csv(dataset); display(df); display(df.groupby(['category','expected_valid']).size())

## Smoke test: 3 intents × 3 pipelines = 9 Together calls

In [ ]:
from src.experiments.run_all import run
smoke=run(dataset,ROOT/'results/raw/smoke.csv',limit=3); display(smoke[['id','pipeline','expected_valid','accepted','predicted_service','latency_ms']])

## Full experiment: 50 × 3 = 150 Together calls

In [ ]:
results=run(dataset,ROOT/'results/raw/experiment_results.csv'); display(results.head())

## Metrics

In [ ]:
from src.metrics import summarize,confusions
metrics=summarize(results); metrics.to_csv(ROOT/'results/metrics/summary.csv',index=False); display(metrics)

## IEEE-ready graphs

In [ ]:
from src.plots.ieee_plots import make_plots
make_plots(metrics,confusions(results),ROOT/'figures')

## Create experiment-1-v2-results.zip

In [ ]:
!cd /content/experiment-1-v2-together && zip -qr /content/experiment-1-v2-results.zip results figures configs data/raw/intents_50.csv
from google.colab import files
files.download('/content/experiment-1-v2-results.zip')